In [15]:
import json
import pandas as pd

In [24]:
df = pd.read_csv('fever_closedbook_qwen2_7b_dev_subset.csv')


In [4]:
df

,id,claim,gold,pred
0,75397,Nikolaj Coster-Waldau worked with the Fox Broa...,SUPPORTS,NOT_ENOUGH_INFO
1,150448,Roman Atwood is a content creator.,SUPPORTS,SUPPORTS
2,214861,"History of art includes architecture, dance, s...",SUPPORTS,SUPPORTS
3,156709,Adrienne Bailon is an accountant.,REFUTES,NOT_ENOUGH_INFO
4,83235,System of a Down briefly disbanded in limbo.,NOT_ENOUGH_INFO,SUPPORTS
...,...,...,...,...
145444,75062,Led Zeppelin released an eponymous debut album...,REFUTES,REFUTES
145445,149256,Taal was romantic.,SUPPORTS,NOT_ENOUGH_INFO
145446,13287,Her stars American actress Rooney Mara.,SUPPORTS,SUPPORTS
145447,13114,J. R. R. Tolkien created Gimli.,SUPPORTS,REFUTES


In [12]:
df.drop(columns='id')

,claim,gold,pred
0,Nikolaj Coster-Waldau worked with the Fox Broa...,SUPPORTS,NOT_ENOUGH_INFO
1,Roman Atwood is a content creator.,SUPPORTS,SUPPORTS
2,"History of art includes architecture, dance, s...",SUPPORTS,SUPPORTS
3,Adrienne Bailon is an accountant.,REFUTES,NOT_ENOUGH_INFO
4,System of a Down briefly disbanded in limbo.,NOT_ENOUGH_INFO,SUPPORTS
...,...,...,...
145444,Led Zeppelin released an eponymous debut album...,REFUTES,REFUTES
145445,Taal was romantic.,SUPPORTS,NOT_ENOUGH_INFO
145446,Her stars American actress Rooney Mara.,SUPPORTS,SUPPORTS
145447,J. R. R. Tolkien created Gimli.,SUPPORTS,REFUTES


In [ ]:
df[['claim','gold','pred']]

## Merging BM25 data

In [25]:
df_open_BM25 = pd.read_csv('llm_classification_results_merged.csv')

In [26]:

all_claims_facts = json.load(open('claim_retrieved_docs_bm25.json', 'r'))
retrieved_bm25 = pd.DataFrame(all_claims_facts)
retrieved_bm25

,claim_id,claim,retrieved_documents
0,0,Nikolaj Coster-Waldau worked with the Fox Broa...,"[{'doc_id': 'Ved_verdens_ende', 'text': 'ved v..."
1,1,Roman Atwood is a content creator.,"[{'doc_id': 'Bedside_Press', 'text': 'bedside ..."
2,2,"History of art includes architecture, dance, s...","[{'doc_id': 'The_arts', 'text': 'the arts is a..."
3,3,Adrienne Bailon is an accountant.,"[{'doc_id': 'Adrienne_Bailon', 'text': 'adrien..."
4,4,System of a Down briefly disbanded in limbo.,"[{'doc_id': 'System_of_a_Down', 'text': 'syste..."
...,...,...,...
145444,145444,Led Zeppelin released an eponymous debut album...,"[{'doc_id': 'Un-Led-Ed', 'text': 'un-led-ed is..."
145445,145445,Taal was romantic.,"[{'doc_id': 'Taal_-LRB-film-RRB-', 'text': 'ta..."
145446,145446,Her stars American actress Rooney Mara.,"[{'doc_id': 'Ann_Mara', 'text': 'ann mara -lrb..."
145447,145447,J. R. R. Tolkien created Gimli.,"[{'doc_id': 'Gimli_-LRB-Middle-earth-RRB-', 't..."


In [31]:


df_open_BM25 = df_open_BM25.rename(columns={
    'classification': 'pred_bm25'
    })

df_open_BM25 = pd.merge(df_open_BM25[['claim', 'pred_bm25']], 
         retrieved_bm25[['claim', 'retrieved_documents']]
        )

df_open_BM25 = df_open_BM25.rename(columns={
    'retrieved_documents': 'retrieved_documents_bm25'
    })

In [32]:
df_open_BM25

,claim,pred_bm25,retrieved_documents_bm25
0,Nikolaj Coster-Waldau worked with the Fox Broa...,SUPPORTS,"[{'doc_id': 'Ved_verdens_ende', 'text': 'ved v..."
1,Roman Atwood is a content creator.,NOT ENOUGH INFO,"[{'doc_id': 'Bedside_Press', 'text': 'bedside ..."
2,Roman Atwood is a content creator.,NOT ENOUGH INFO,"[{'doc_id': 'Bedside_Press', 'text': 'bedside ..."
3,Roman Atwood is a content creator.,NOT ENOUGH INFO,"[{'doc_id': 'Bedside_Press', 'text': 'bedside ..."
4,"History of art includes architecture, dance, s...",SUPPORTS,"[{'doc_id': 'The_arts', 'text': 'the arts is a..."
...,...,...,...
188662,Taal was romantic.,SUPPORTS,"[{'doc_id': 'Taal_-LRB-film-RRB-', 'text': 'ta..."
188663,Her stars American actress Rooney Mara.,SUPPORTS,"[{'doc_id': 'Ann_Mara', 'text': 'ann mara -lrb..."
188664,J. R. R. Tolkien created Gimli.,SUPPORTS,"[{'doc_id': 'Gimli_-LRB-Middle-earth-RRB-', 't..."
188665,Susan Sarandon is an award winner.,SUPPORTS,[{'doc_id': 'List_of_awards_and_nominations_re...


## Merging dense data

In [ ]:
# TODO

## Combine ALL

In [36]:
df_combined = pd.merge(df[['claim','gold','pred']], 
                       df_open_BM25[['claim', 'pred_bm25', 'retrieved_documents_bm25']]
                      )

In [45]:
df_combined['pred_bm25'] = df_combined['pred_bm25'].replace('NOT ENOUGH INFO','NOT_ENOUGH_INFO')

In [52]:
df_combined['pred_bm25'] = df_combined['pred_bm25'].fillna('NOT_ENOUGH_INFO')

In [46]:
df_combined

,claim,gold,pred,pred_bm25,retrieved_documents_bm25
0,Nikolaj Coster-Waldau worked with the Fox Broa...,SUPPORTS,NOT_ENOUGH_INFO,SUPPORTS,"[{'doc_id': 'Ved_verdens_ende', 'text': 'ved v..."
1,Roman Atwood is a content creator.,SUPPORTS,SUPPORTS,NOT_ENOUGH_INFO,"[{'doc_id': 'Bedside_Press', 'text': 'bedside ..."
2,Roman Atwood is a content creator.,SUPPORTS,SUPPORTS,NOT_ENOUGH_INFO,"[{'doc_id': 'Bedside_Press', 'text': 'bedside ..."
3,Roman Atwood is a content creator.,SUPPORTS,SUPPORTS,NOT_ENOUGH_INFO,"[{'doc_id': 'Bedside_Press', 'text': 'bedside ..."
4,Roman Atwood is a content creator.,SUPPORTS,SUPPORTS,NOT_ENOUGH_INFO,"[{'doc_id': 'Bedside_Press', 'text': 'bedside ..."
...,...,...,...,...,...
512344,J. R. R. Tolkien created Gimli.,SUPPORTS,REFUTES,SUPPORTS,"[{'doc_id': 'Gimli_-LRB-Middle-earth-RRB-', 't..."
512345,Susan Sarandon is an award winner.,SUPPORTS,SUPPORTS,SUPPORTS,[{'doc_id': 'List_of_awards_and_nominations_re...
512346,Susan Sarandon is an award winner.,SUPPORTS,SUPPORTS,SUPPORTS,[{'doc_id': 'List_of_awards_and_nominations_re...
512347,Susan Sarandon is an award winner.,SUPPORTS,SUPPORTS,SUPPORTS,[{'doc_id': 'List_of_awards_and_nominations_re...


## Classification result

In [62]:
df_combined_clean = df_combined.drop_duplicates('claim')

### Close book

In [63]:
from sklearn.metrics import classification_report


print(classification_report(df_combined_clean['gold'].to_list(), 
                            df_combined_clean['pred'].to_list()))



                 precision    recall  f1-score   support

NOT_ENOUGH_INFO       0.39      0.62      0.48     33567
        REFUTES       0.48      0.34      0.40     28171
       SUPPORTS       0.81      0.68      0.74     73807

       accuracy                           0.59    135545
      macro avg       0.56      0.55      0.54    135545
   weighted avg       0.64      0.59      0.60    135545



### Open book (BM25)

In [64]:
from sklearn.metrics import classification_report


print(classification_report(df_combined_clean['gold'].to_list(), 
                            df_combined_clean['pred_bm25'].to_list()))


                 precision    recall  f1-score   support

NOT_ENOUGH_INFO       0.47      0.38      0.42     33567
        REFUTES       0.51      0.42      0.46     28171
       SUPPORTS       0.74      0.85      0.79     73807

       accuracy                           0.65    135545
      macro avg       0.57      0.55      0.56    135545
   weighted avg       0.63      0.65      0.63    135545



### Open book (dense)

In [ ]:
# TODO

## Analysis

In [ ]:
## cases where close book are good, but open book failed (WEIRD)

In [65]:
mask = (
    (df_combined_clean['gold']==df_combined_clean['pred']) &
    (df_combined_clean['gold']!=df_combined_clean['pred_bm25'])

)

In [67]:
df_weird = df_combined_clean[mask]

In [71]:
df_weird[(
    (df_weird['pred']=='SUPPORTS')
)]['pred_bm25'].value_counts()

pred_bm25
NOT_ENOUGH_INFO    2130
REFUTES            1295
Name: count, dtype: int64